In [37]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error, accuracy_score
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib
import tldextract

In [38]:
df = pd.read_csv('csvs/urldata.csv')
df.drop(columns=['Unnamed: 0'],inplace=True)

In [39]:
print(df.columns)

Index(['url', 'label', 'result'], dtype='object')


In [40]:
df.head()

,url,label,result
0,https://www.google.com,benign,0
1,https://www.youtube.com,benign,0
2,https://www.facebook.com,benign,0
3,https://www.baidu.com,benign,0
4,https://www.wikipedia.org,benign,0


In [41]:
risk_words = ['login','admin','update','secure','account']

In [42]:
def get_features(url):
    extracted_object = tldextract.extract(url)
    size = len(url)
    https = url.startswith('https://')
    subdomain = extracted_object.subdomain
    #domain = extracted_object.domain
    suffix = extracted_object.suffix
    dots = url.count('.')
    dashes = url.count('-')
    numbers = sum(c.isdigit() for c in url)
    risky=0
    for d in risk_words:
        if d in url:
            risky+=1
    f_slash = sum(c == '/' for c in url)
    b_slash = sum(c == '\\' for c in url)
    underscore = sum(c == '_' for c in url)
    at = True if '@' in url else False
    #return size, https, subdomain, suffix, dots, dashes, numbers, login, f_slash, b_slash, underscore, at
    return {
        #'size' : size,
        'https' : https,
        #'subdomain' : subdomain,
        'suffix' : suffix,
        'dots' : dots,
        'dashes' : dashes,
        'numbers' : numbers,
        'risky' : risky,
        'f_slash' : f_slash,
        'b_slash' : b_slash,
        'underscore' : underscore,
        'at' : at
    }
    

In [43]:
df_features = df['url'].apply(get_features).apply(pd.Series)

In [44]:
final_df = pd.concat([df,df_features], axis=1)

In [45]:
final_df.head()

,url,label,result,https,suffix,dots,dashes,numbers,risky,f_slash,b_slash,underscore,at
0,https://www.google.com,benign,0,True,com,2,0,0,0,2,0,0,False
1,https://www.youtube.com,benign,0,True,com,2,0,0,0,2,0,0,False
2,https://www.facebook.com,benign,0,True,com,2,0,0,0,2,0,0,False
3,https://www.baidu.com,benign,0,True,com,2,0,0,0,2,0,0,False
4,https://www.wikipedia.org,benign,0,True,org,2,0,0,0,2,0,0,False


In [46]:
final_df.columns

Index(['url', 'label', 'result', 'https', 'suffix', 'dots', 'dashes',
       'numbers', 'risky', 'f_slash', 'b_slash', 'underscore', 'at'],
      dtype='object')

In [49]:
numerical_cols = ['https', 'dots', 'dashes', 'numbers', 'risky', 'f_slash', 'b_slash', 'underscore', 'at']
categorical_cols = ['suffix']


In [50]:
y = final_df['result']
X = final_df[numerical_cols + categorical_cols]

In [51]:
imputer = Pipeline([('imputer', SimpleImputer())])
encoder = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), 
                     ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
ordinal = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), 
                     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])

In [57]:
preprocessor = ColumnTransformer([('imputer', imputer, numerical_cols), 
                                  ('encoder', encoder, ['suffix'])])

In [58]:
model = XGBClassifier(random_state=1, n_estimators=100,n_jobs=-1, max_depth=3, min_child_weight=10, learning_rate=0.05)

In [59]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

In [60]:
final_pipeline = Pipeline([('preprocessor',preprocessor), ('model', model)])

In [61]:
cross_validate(final_pipeline, X, y, cv=kf, return_train_score=True)

{'fit_time': array([11.23590231,  8.40867066,  9.28859758,  8.85263515,  8.93973494]),
 'score_time': array([0.63690996, 0.65981746, 0.80357552, 0.78005528, 0.81719518]),
 'test_score': array([0.98840464, 0.98905981, 0.98921531, 0.98864886, 0.98939301]),
 'train_score': array([0.9890015 , 0.98892101, 0.98911537, 0.98919034, 0.98899042])}

In [ ]:
joblib.dump('final pipeline', final_pipeline)

['model.pkl']